# La cassetta degli attrezzi: espressioni regolari, normalizzazione e distanza di edit

Il codice del capitolo [«La cassetta degli attrezzi: espressioni regolari, normalizzazione e distanza di edit»](https://book.paithon.it/main/NaturalLanguageProcessing/strumenti-classici.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy scikit-learn torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## La cassetta degli attrezzi: espressioni regolari, normalizzazione e distanza di edit

[Leggi la pagina](https://book.paithon.it/main/NaturalLanguageProcessing/strumenti-classici.html)


### Le espressioni regolari: descrivere uno schema, non una parola


In [ ]:
import re

testo = ("Il gatto nero salta sul muro di via dei Tigli 42. La gatta lo "
         "guarda dal balcone: CAP 95125, visita dal veterinario il "
         "3/7/2026 alle 18:30.")

# tutte le forme di "gatto": la radice gatt- più una vocale finale
re.findall(r"\bgatt[oaie]\b", testo)
# ['gatto', 'gatta']

# il CAP: esattamente cinque cifre isolate
re.findall(r"\b\d{5}\b", testo)
# ['95125']

# una data giorno/mese/anno
re.findall(r"\b\d{1,2}/\d{1,2}/\d{4}\b", testo)
# ['3/7/2026']

# gruppi: catturare giorno, mese e anno separatamente
m = re.search(r"(\d{1,2})/(\d{1,2})/(\d{4})", testo)
m.group(1), m.group(2), m.group(3)
# ('3', '7', '2026')

### Normalizzare il testo: decidere cosa è «la stessa parola»


In [ ]:
import re
import unicodedata

STOPWORD = {"il", "lo", "la", "i", "gli", "le", "un", "una", "di", "a",
            "da", "in", "su", "sul", "per", "con", "e", "che", "è"}

def normalizza(testo):
    testo = unicodedata.normalize("NFKC", testo)  # codifiche Unicode uniformi
    testo = testo.lower()                         # tutto minuscolo
    testo = re.sub(r"[^\w\s]", " ", testo)        # via la punteggiatura
    return [p for p in testo.split() if p not in STOPWORD]

normalizza("Il gatto NERO salta sul muro!")
# ['gatto', 'nero', 'salta', 'muro']

### La distanza di edit: quante mosse da una parola all'altra


In [ ]:
def levenshtein(a, b):
    prec = list(range(len(b) + 1))          # riga dei casi base D[0][j] = j
    for i, ca in enumerate(a, start=1):
        cur = [i]                           # caso base D[i][0] = i
        for j, cb in enumerate(b, start=1):
            costo = 0 if ca == cb else 1
            cur.append(min(prec[j] + 1,          # cancellazione
                           cur[j - 1] + 1,       # inserzione
                           prec[j - 1] + costo)) # sostituzione o lettera uguale
        prec = cur
    return prec[-1]

levenshtein("muro", "mare")   # 2
levenshtein("carta", "casa")  # 2
levenshtein("gatot", "gatto") # 2

## Rappresentare il testo: dai token agli embedding

[Leggi la pagina](https://book.paithon.it/main/NaturalLanguageProcessing/rappresentare-testo.html)


### Contare le parole: bag-of-words e TF-IDF


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = ["il gatto nero salta sul muro",
          "il cane dorme sul divano"]

vec = TfidfVectorizer()
X = vec.fit_transform(corpus)   # matrice sparsa documenti x vocabolario
print(vec.get_feature_names_out())  # il vocabolario appreso
print(X.toarray())                  # pesi TF-IDF per ciascun documento

### Dalla parola alla frase: gli embedding di frase


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# Uno spazio di partenza che NON separa gli argomenti: quattro argomenti,
# trentadue "frasi" ciascuno, coordinate casuali. È il caso di un encoder
# che non è mai stato addestrato alla somiglianza.
N_ARG, PER_ARG, D = 4, 32, 16
argomento = torch.arange(N_ARG).repeat_interleave(PER_ARG)
grezzi = torch.randn(N_ARG * PER_ARG, D)
indici = [torch.nonzero(argomento == k).flatten() for k in range(N_ARG)]

def coseni(V):
    """Coseno medio dentro l'argomento e fra argomenti diversi."""
    V = F.normalize(V, dim=-1)
    S = V @ V.T
    stesso = argomento[:, None] == argomento[None, :]
    stesso.fill_diagonal_(False)
    return S[stesso].mean().item(), S[~stesso].mean().item()

def sorteggia(n):
    """Una tripletta per riga: ancora, un simile, un diverso."""
    a = torch.randint(0, len(grezzi), (n,))
    ka = argomento[a]
    p = torch.stack([indici[k][torch.randint(0, PER_ARG, (1,))][0] for k in ka])
    kn = (ka + torch.randint(1, N_ARG, (n,))) % N_ARG      # un argomento diverso
    neg = torch.stack([indici[k][torch.randint(0, PER_ARG, (1,))][0] for k in kn])
    return a, p, neg

# La "torre": UNA sola rete, applicata a tutti e tre gli ingressi.
# I pesi condivisi sono ciò che rende la rete siamese.
torre = nn.Sequential(nn.Linear(D, 32), nn.ReLU(), nn.Linear(32, D))
ott = torch.optim.Adam(torre.parameters(), lr=1e-2)
MARGINE = 0.3

for _ in range(400):
    a, p, neg = sorteggia(128)
    A, P, N = (F.normalize(torre(grezzi[i]), dim=-1) for i in (a, p, neg))
    # l'ancora deve stare più vicina al positivo che al negativo, e non di
    # poco: almeno di un margine. Chi già rispetta il margine non contribuisce.
    perdita = F.relu(MARGINE - (A * P).sum(-1) + (A * N).sum(-1)).mean()
    ott.zero_grad(); perdita.backward(); ott.step()

for etichetta, V in [("prima", grezzi), ("dopo ", torre(grezzi).detach())]:
    dentro, fuori = coseni(V)
    print(f"{etichetta}: coseno dentro l'argomento {dentro:+.3f}, "
          f"fra argomenti diversi {fuori:+.3f}, distacco {dentro - fuori:+.3f}")

## Come si spezza il testo: il Byte Pair Encoding

[Leggi la pagina](https://book.paithon.it/main/NaturalLanguageProcessing/tokenizzatori.html)


### Trenta righe di Python


In [ ]:
from collections import Counter

# corpus giocattolo: parola -> quante volte compare
corpus = {"basso": 6, "bassotto": 2, "bosso": 3, "rosso": 9, "rossetto": 5}


def conta_coppie(pezzi, corpus):
    """Frequenza di ogni coppia adiacente, pesata sulle occorrenze della parola."""
    coppie = Counter()
    for parola, simboli in pezzi.items():
        for coppia in zip(simboli, simboli[1:]):
            coppie[coppia] += corpus[parola]
    return coppie


def fondi(simboli, coppia):
    """Sostituisce ogni occorrenza della coppia con il simbolo unito."""
    uniti, i = [], 0
    while i < len(simboli):
        if i < len(simboli) - 1 and (simboli[i], simboli[i + 1]) == coppia:
            uniti.append(simboli[i] + simboli[i + 1])
            i += 2
        else:
            uniti.append(simboli[i])
            i += 1
    return tuple(uniti)


def addestra(corpus, n_fusioni):
    pezzi = {parola: tuple(parola) for parola in corpus}   # si parte dai caratteri
    fusioni = []
    for _ in range(n_fusioni):
        coppie = conta_coppie(pezzi, corpus)
        if not coppie:
            break
        # la piu' frequente; a parita' di conteggio, la prima in ordine alfabetico
        coppia = min(coppie, key=lambda c: (-coppie[c], c))
        fusioni.append(coppia)
        pezzi = {p: fondi(s, coppia) for p, s in pezzi.items()}
    return fusioni


def tokenizza(parola, fusioni):
    """Riapplica le fusioni imparate, nello stesso ordine."""
    simboli = tuple(parola)
    for coppia in fusioni:
        simboli = fondi(simboli, coppia)
    return simboli


iniziali = {parola: tuple(parola) for parola in corpus}
print("coppie al primo passo:", conta_coppie(iniziali, corpus).most_common(5))

fusioni = addestra(corpus, 10)
for i, (a, b) in enumerate(fusioni, 1):
    print(f"{i:2d}. {a} + {b} -> {a + b}")

print("bassetto   ->", tokenizza("bassetto", fusioni))
print("rossellini ->", tokenizza("rossellini", fusioni))

## Insegnare a giudicare: classificare il testo

[Leggi la pagina](https://book.paithon.it/main/NaturalLanguageProcessing/classificazione-testo.html)


### Alla prova: il sentiment delle recensioni


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

recensioni = [
    "un capolavoro, attori straordinari e regia impeccabile",
    "film splendido, mi ha emozionato dall'inizio alla fine",
    "divertente e intelligente, lo rivedrei subito",
    "una storia che sorprende, fotografia bellissima",
    "che noia, due ore interminabili e senza idee",
    "recitazione pessima e trama piena di buchi",
    "una delusione totale, soldi buttati",
    "banale e prevedibile, mi sono addormentato",
]
etichette = [1, 1, 1, 1, 0, 0, 0, 0]  # 1 = positiva, 0 = negativa

# CountVectorizer = bag-of-words; alpha=1.0 e' lo smoothing di Laplace
modello = make_pipeline(CountVectorizer(), MultinomialNB(alpha=1.0))
modello.fit(recensioni, etichette)

nuove = ["una regia splendida e attori bravissimi",
         "prevedibile e senza emozioni, che delusione"]
print(modello.predict(nuove))          # [1 0]
print(modello.predict_proba(nuove))    # probabilita' per classe

### Il classificatore in PyTorch


In [ ]:
import torch
from torch import nn
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer()
X = torch.tensor(vec.fit_transform(recensioni).toarray(), dtype=torch.float32)
y = torch.tensor(etichette, dtype=torch.float32).unsqueeze(1)

modello = nn.Linear(X.shape[1], 1)      # un peso per parola, piu' il bias
loss_fn = nn.BCEWithLogitsLoss()        # sigmoide + cross-entropia binaria
ottim = torch.optim.Adam(modello.parameters(), lr=0.05)

for epoca in range(300):
    ottim.zero_grad()
    perdita = loss_fn(modello(X), y)    # il modello emette logit, non probabilita'
    perdita.backward()
    ottim.step()

with torch.no_grad():
    X_nuove = torch.tensor(vec.transform(nuove).toarray(), dtype=torch.float32)
    print(torch.sigmoid(modello(X_nuove)).squeeze())  # probabilita' "positiva"

In [ ]:
pesi = modello.weight.detach().squeeze()
parole = vec.get_feature_names_out()
ordine = pesi.argsort().tolist()
print("piu' negative:", [parole[i] for i in ordine[:3]])
print("piu' positive:", [parole[i] for i in ordine[-3:]])

## Scommettere sulla prossima parola: i modelli n-gram

[Leggi la pagina](https://book.paithon.it/main/NaturalLanguageProcessing/modelli-ngram.html)


### Un bigramma in trenta righe di Python


In [ ]:
import math
import random
from collections import Counter, defaultdict

corpus = [
    "il gatto nero salta sul muro",
    "il gatto bianco dorme sul divano",
    "il cane guarda il gatto nero",
]

INIZIO, FINE = "<s>", "</s>"

# 1. Conteggi: conta[w1][w2] = quante volte w2 segue w1
conta = defaultdict(Counter)
for frase in corpus:
    parole = [INIZIO] + frase.split() + [FINE]
    for w1, w2 in zip(parole, parole[1:]):
        conta[w1][w2] += 1

vocabolario = {w for frase in corpus for w in frase.split()} | {FINE}
V = len(vocabolario)                      # 12: 11 parole + </s>

# 2. Probabilita': massima verosimiglianza e Laplace
def p_mle(w1, w2):
    tot = sum(conta[w1].values())
    return conta[w1][w2] / tot if tot else 0.0

def p_laplace(w1, w2):
    return (conta[w1][w2] + 1) / (sum(conta[w1].values()) + V)

print(p_mle("il", "gatto"))       # 0.75
print(p_mle("cane", "nero"))      # 0.0 -> lo zero che azzera tutto
print(p_laplace("cane", "nero"))  # 0.0769... -> piccola ma viva

# 3. Generazione: una passeggiata di scommesse da <s> a </s>
def genera(seme):
    rng = random.Random(seme)
    parola, frase = INIZIO, []
    while len(frase) < 20:
        seguiti = conta[parola]
        parola = rng.choices(list(seguiti), weights=seguiti.values())[0]
        if parola == FINE:
            break
        frase.append(parola)
    return " ".join(frase)

for seme in range(3):
    print(genera(seme))
# il cane guarda il gatto nero
# il cane guarda il gatto nero
# il cane guarda il cane guarda il gatto nero

# 4. Perplessita' di una frase secondo il modello lisciato
def perplessita(frase):
    parole = [INIZIO] + frase.split() + [FINE]
    log2p = sum(math.log2(p_laplace(w1, w2))
                for w1, w2 in zip(parole, parole[1:]))
    return 2 ** (-log2p / (len(parole) - 1))

# nessuna delle tre e' nel corpus di addestramento: si valuta su testo nuovo
print(perplessita("il gatto nero salta sul divano"))  # ~5.5  tutte coppie viste
print(perplessita("il cane nero salta sul divano"))   # ~7.0  una coppia mai vista
print(perplessita("divano sul salta nero gatto il"))  # ~14.2 la prima, rimescolata

## Modelli di sequenza: da RNN ai Transformer

[Leggi la pagina](https://book.paithon.it/main/NaturalLanguageProcessing/modelli-sequenza.html)


### In pratica, con PyTorch


In [ ]:
import torch
from torch import nn

class ClassificatoreSentiment(nn.Module):
    def __init__(self, vocab=10000, dim=64, hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab, dim)  # parola -> vettore
        self.rnn = nn.LSTM(dim, hidden, batch_first=True)  # prova nn.RNN o nn.GRU
        self.out = nn.Linear(hidden, 2)            # 2 classi: negativo/positivo

    def forward(self, x):          # x: (batch, lunghezza), indici di parole
        e = self.embedding(x)      # (batch, lunghezza, dim)
        h, _ = self.rnn(e)         # stato nascosto a ogni passo
        return self.out(h[:, -1])  # ultimo passo -> logit per CrossEntropyLoss

## Da frase a frase: tradurre con le reti

[Leggi la pagina](https://book.paithon.it/main/NaturalLanguageProcessing/seq2seq-traduzione.html)


### Leggere in due direzioni


In [ ]:
import torch
from torch import nn

lstm = nn.LSTM(
    input_size=64, hidden_size=128,
    num_layers=2,        # due strati impilati
    bidirectional=True,  # lettura in entrambe le direzioni
    batch_first=True,
)

x = torch.randn(1, 6, 64)  # 1 frase, 6 parole ("Il gatto nero salta sul muro")
out, (h, c) = lstm(x)
print(out.shape)  # torch.Size([1, 6, 256]): 2 direzioni x 128 per ogni parola
print(h.shape)    # torch.Size([4, 1, 128]): 2 strati x 2 direzioni

### Con la soluzione accanto: il teacher forcing


In [ ]:
import numpy as np

V, T, N = 10, 60, 20000   # 10 parole, frasi lunghe 60, 20000 frasi per volta
FUGA = 0.01               # sui paia che i dati contengono sbaglia una volta su cento

def modello(sa_continuare_fuori):
    """Le probabilità della parola dopo, dato il paio che precede."""
    M = np.full((V, V, V), 1.0 / V)      # paia mai viste: il modello tira a caso
    for a in range(V):
        paia = range(V) if sa_continuare_fuori else [(a + 1) % V]
        for b in paia:
            M[a, b] = FUGA / (V - 1)
            M[a, b, (b + 1) % V] = 1 - FUGA
    return M.cumsum(axis=2)

def scrivi(cum, da_se, seme=20260830):
    """N frasi; se `da_se` è falso, prima di ogni parola torna l'inizio giusto."""
    rng = np.random.default_rng(seme)
    seq = np.zeros((N, T), dtype=int)
    seq[:, 1] = 1
    scritte = np.zeros((N, T), dtype=int)
    for t in range(2, T):
        a, b = (seq[:, t-2], seq[:, t-1]) if da_se else ((t-2) % V, (t-1) % V)
        scelta = (cum[a, b] < rng.random(N)[:, None]).sum(axis=1).clip(0, V - 1)
        scritte[:, t] = scelta
        seq[:, t] = scelta if da_se else t % V
    return seq, scritte

reale = modello(False)
_, con_soluzione = scrivi(reale, da_se=False)
libere, _ = scrivi(reale, da_se=True)
ideali, _ = scrivi(modello(True), da_se=True)
ok_sol = con_soluzione[:, 2:] == np.arange(2, T) % V
ok_lib = libere[:, 2:] == (libere[:, 1:-1] + 1) % V
ok_ide = ideali[:, 2:] == (ideali[:, 1:-1] + 1) % V

print("parola  con la soluzione   da sé   da sé, sulle frasi ancora intatte   frasi intatte")
for t in (0, 8, 28, 57):
    intatte = ok_lib[:, :t].all(axis=1)      # nessun errore prima di questa parola
    print(f"{t+2:6}{100*ok_sol[:, t].mean():16.1f}%{100*ok_lib[:, t].mean():8.1f}%"
          f"{100*ok_lib[intatte, t].mean():36.1f}%{100*intatte.mean():16.1f}%")

print()
print("controprova, con un modello che sappia continuare anche fuori strada:")
print("  da sé      " + "  ".join(f"{100*ok_ide[:, t].mean():.1f}%" for t in (0, 8, 28, 57)))
print("  frasi intatte " + "  ".join(
    f"{100*ok_ide[:, :t].all(axis=1).mean():.1f}%" for t in (0, 8, 28, 57)))

## Un'etichetta per ogni parola: POS tagging e riconoscimento di entità

[Leggi la pagina](https://book.paithon.it/main/NaturalLanguageProcessing/etichettare-sequenze.html)


### La via neurale: una BiLSTM per etichettare


In [ ]:
import torch
from torch import nn

class TaggerBiLSTM(nn.Module):
    def __init__(self, vocab=10000, num_tag=17, dim=64, hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab, dim)   # parola -> vettore
        self.lstm = nn.LSTM(dim, hidden, batch_first=True,
                            bidirectional=True)     # legge nei due sensi
        self.out = nn.Linear(2 * hidden, num_tag)   # un logit per etichetta

    def forward(self, x):          # x: (batch, lunghezza), indici di parole
        e = self.embedding(x)      # (batch, lunghezza, dim)
        h, _ = self.lstm(e)        # (batch, lunghezza, 2*hidden)
        return self.out(h)         # logit per OGNI parola, non solo l'ultima

*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

modello = TaggerBiLSTM()
perdita = nn.CrossEntropyLoss(ignore_index=-100)  # -100 = padding da ignorare

# frasi: (batch, lunghezza), indici di parole; tag: stessa forma, -100 sul padding
logits = modello(frasi)                    # (batch, lunghezza, 17)
loss = perdita(logits.reshape(-1, 17),     # una riga per token
               tag.reshape(-1))            # un'etichetta per token
loss.backward()                            # poi optimizer.step(), come sempre
```


## La struttura nascosta della frase: sintassi e parsing

[Leggi la pagina](https://book.paithon.it/main/NaturalLanguageProcessing/struttura-frase.html)


### Contare le letture in trenta righe di Python


In [ ]:
# Grammatica giocattolo in forma normale di Chomsky (6 regole + lessico)
lessico = {
    "ho": {"AUX"}, "visto": {"V"}, "un": {"DET"}, "il": {"DET"},
    "uomo": {"N"}, "binocolo": {"N"}, "cappello": {"N"},
    "gatto": {"N"}, "con": {"P"},
}
regole = [                    # A -> B C
    ("SN", "DET", "N"),       # "un uomo", "il binocolo"
    ("SP", "P",   "SN"),      # "con il binocolo"
    ("SN", "SN",  "SP"),      # attacco al nome: l'uomo HA il binocolo
    ("SV", "V",   "SN"),      # "visto un uomo"
    ("SV", "SV",  "SP"),      # attacco al verbo: ho guardato COL binocolo
    ("F",  "AUX", "SV"),      # "ho" + sintagma verbale
]

def conta_alberi(parole):
    n = len(parole)
    # tab[i][j] = {categoria: quanti alberi coprono parole[i:j]}
    tab = [[{} for _ in range(n + 1)] for _ in range(n + 1)]
    for i, w in enumerate(parole):
        for cat in lessico[w]:
            tab[i][i + 1][cat] = 1
    for lung in range(2, n + 1):          # intervalli dal corto al lungo
        for i in range(n - lung + 1):
            j = i + lung
            for k in range(i + 1, j):     # punto di taglio
                for A, B, C in regole:
                    if B in tab[i][k] and C in tab[k][j]:
                        tab[i][j][A] = (tab[i][j].get(A, 0)
                                        + tab[i][k][B] * tab[k][j][C])
    return tab[0][n].get("F", 0)

print(conta_alberi("ho visto un uomo con il binocolo".split()))        # 2
print(conta_alberi(
    "ho visto un uomo con il binocolo con il cappello".split()))       # 5

## Parlare con le macchine: dialogo e chatbot

[Leggi la pagina](https://book.paithon.it/main/NaturalLanguageProcessing/dialogo-chatbot.html)


### Lo specchio di regole: dentro ELIZA


In [ ]:
import re

# pronomi e verbi da "riflettere": la prospettiva passa da io a tu
RIFLESSI = {"mio": "tuo", "mia": "tua", "miei": "tuoi", "mie": "tue",
            "mi": "ti", "me": "te", "io": "tu",
            "sono": "sei", "ho": "hai", "posso": "puoi", "voglio": "vuoi"}

# coppie (schema, risposta): {0} e' il pezzo di frase catturato dallo schema
REGOLE = [
    (r"mi sento (.+)",            "Da quanto tempo ti senti {0}?"),
    (r"vorrei (.+)",              "Perché vorresti {0}?"),
    (r"(?:penso|credo) che (.+)", "E cosa ti fa credere che {0}?"),
    (r"tutti (.+)",               "Proprio tutti {0}? Nessuna eccezione?"),
    (r"(.+)\?",                   "Perché me lo chiedi?"),
]

def rifletti(testo):
    """Riscrive un frammento dal punto di vista dell'interlocutore."""
    return " ".join(RIFLESSI.get(parola, parola) for parola in testo.split())

def eliza(frase):
    frase = frase.lower().strip(" .!")
    for schema, risposta in REGOLE:       # la prima regola che aggancia vince
        match = re.match(schema, frase)
        if match:
            return risposta.format(*map(rifletti, match.groups()))
    return "Capisco. Vai avanti."         # ripiego: nessuno schema ha agganciato

for battuta in ["Mi sento in colpa verso mia sorella",
                "Penso che nessuno mi ascolti",
                "Il gatto nero salta sul muro"]:
    print("TU:   ", battuta)
    print("ELIZA:", eliza(battuta))

# TU:    Mi sento in colpa verso mia sorella
# ELIZA: Da quanto tempo ti senti in colpa verso tua sorella?
# TU:    Penso che nessuno mi ascolti
# ELIZA: E cosa ti fa credere che nessuno ti ascolti?
# TU:    Il gatto nero salta sul muro
# ELIZA: Capisco. Vai avanti.